# Tutorial 02: Data Management and HDF5 Handling

Astronomical datasets, especially sky maps and simulation boxes, can be very large. `hifigps` uses the HDF5 format for efficient storage and providing easy-to-use wrappers for data I/O and metadata management.

This tutorial covers saving, reading, inspecting, and manipulating HDF5 files using the `hifigps.data` module.

In [1]:
import numpy as np
from hifigps.data import (
    is_exist, get_file_dir, get_filename, delete_files,
    vitems, del_data, read_h5, save_h5, save_attrs,
    load_attrs, split_data_generator
)

## 1. Creating and Saving Data

Let's create some dummy data and save it to an HDF5 file.

In [2]:
np.random.seed(42)
test_data = np.random.randn(10, 10, 10)
filepath = "tutorial_data.h5"

# Check if file already exists
if not is_exist(filepath):
    # save_h5 takes a list of keys and a list of data arrays
    save_h5(filepath, ["map_data"], [test_data])
    print(f"Data saved to {filepath}")

## 2. Inspecting HDF5 File Structure

The `vitems` function allows you to "visit" every item in the file and print its structure (shape, dtype, attributes).

In [3]:
vitems(filepath)

## 3. Metadata Management (Attributes)

Keeping track of units, authors, and processing steps is crucial for scientific reproducibility.

In [4]:
file_attrs = {
    "author": "Gemini CLI",
    "version": "1.0",
    "description": "Tutorial example file"
}

dataset_attrs = {
    "map_data": {"units": "mK", "info": "Simulated HI signal"}
}

save_attrs(filepath, file_attrs, dataset_attrs)

# Load and verify attributes
attrs = load_attrs(filepath)
print("Loaded Attributes:")
import json
print(json.dumps(attrs, indent=2))

## 4. Reading Data with Attributes

You can read data and optionally print its attributes.

In [5]:
data_read = read_h5(filepath, "map_data", print_attrs=True)
print(f"Shape of data read: {data_read.shape}")

## 5. Batch Processing with Generators

For very large catalogs, `split_data_generator` helps in processing data in chunks to save memory.

In [6]:
# Split the 10-element data into batches of size 3
for i, (batch,) in enumerate(split_data_generator(3, data_read)):
    print(f"Batch {i} size: {batch.shape[0]}")

## 6. File and Data Cleanup

Utilities for deleting specific datasets or entire files.

In [7]:
# Delete specific dataset within file
del_data(filepath, "map_data")

# Check content again
print("Content after deletion:")
vitems(filepath)

# Delete the entire file
delete_files([filepath])